# 06 · Experiment E (explainability), Experiment F (robustness), uncertainty & the final explainable PE map

**Explanations compared:** Grad-CAM, Grad-CAM++, vessel attention (token attention painted onto the vessel graph), the model's localisation heat-map, and the **proposed combined map** `norm(heat × (½·GradCAM + ½·vessel-attention)) ∩ vessel`.

Faithfulness/plausibility metrics available without voxel ground truth: **deletion score** (drop in PE probability when the top-p % voxels are removed — lower is more faithful), **z-hit rate** (map peak falls on a PE-labelled slice ±1), and **vessel mass** (fraction of map mass inside the vessel mask).

In [ ]:
%run ./00_common.ipynb

In [ ]:
sp = load_splits(); tr, va, te = sp["train"], sp["val"], sp["test"]
TH = json.load(open(f"{cfg.ckpt_dir}/thresholds.json")); cfg.thr_det, cfg.thr_slice = TH["thr_det"], TH["thr_slice"]
model = PENet(cfg, Flags()).to(DEVICE); model.load_state_dict(torch.load(f"{cfg.ckpt_dir}/penet_full.pt", map_location=DEVICE)); model.eval()

import pandas as pd, matplotlib.pyplot as plt
from scipy import ndimage as ndi
norm01 = lambda x: (x - x.amin((1, 2, 3, 4), keepdim=True)) / (x.amax((1, 2, 3, 4), keepdim=True) - x.amin((1, 2, 3, 4), keepdim=True) + 1e-8)
up = lambda x, like: F.interpolate(x, size=like.shape[2:], mode="trilinear", align_corners=False)

def gradcam3d(model, b, plus=False):
    model.eval()
    with torch.enable_grad():
        out = model(b, keep_feat=True); f = out["f4"]; g = torch.autograd.grad(out["det_logit"].sum(), f, retain_graph=False)[0]
    if plus:
        g2, g3 = g ** 2, g ** 3; alpha = g2 / (2 * g2 + f.sum((2, 3, 4), keepdim=True) * g3 + 1e-7); w = (alpha * F.relu(g)).sum((2, 3, 4), keepdim=True)
    else: w = g.mean((2, 3, 4), keepdim=True)
    return norm01(up(F.relu((w * f).sum(1, keepdim=True)), b["vol"]).detach())

@torch.no_grad()
def explain(model, b):
    out = model(b); vd = F.max_pool3d(b["vessel"], 3, 1, 1)
    low = tuple(max(n // 4, 2) for n in b["vol"].shape[2:]); imp = out["tok_imp"] / out["tok_imp"].amax(1, keepdim=True).clamp(min=1e-8)
    attn = norm01(up(gaussian_field(b["centers"], imp * b["valid"].float(), low, cfg.gate_sigma), b["vol"])) * vd
    heat = out["heat"].float()
    cam = gradcam3d(model, b); campp = gradcam3d(model, b, plus=True)
    comb = norm01(heat * (0.5 * cam + 0.5 * attn)) * vd
    return dict(gradcam=cam * vd, gradcampp=campp * vd, attention=attn, heat=norm01(heat), combined=comb), out

## Experiment E — compare explanation methods

In [ ]:
@torch.no_grad()
def det_prob(b): return torch.sigmoid(model(b)["det_logit"].float())

def deletion_score(b, emap, fracs=(0.0, 0.02, 0.05, 0.1, 0.2), fill=0.45):
    flat = emap.flatten(1); ps = []
    for f in fracs:
        k = max(int(f * flat.shape[1]), 1); thr = flat.topk(k, 1).values[:, -1:]
        mask = (emap >= thr.view(-1, 1, 1, 1, 1)) if f > 0 else torch.zeros_like(emap, dtype=torch.bool)
        bb = dict(b); bb["vol"] = torch.where(mask, torch.full_like(b["vol"], fill), b["vol"]); ps.append(float(det_prob(bb)))
    return float(np.mean(ps))

pos_ids = [s for s in te if np.load(f"{cfg.vol_dir}/{s}.npz")["y"][0] == 1][:40]; rows = []
for sid in pos_ids:
    b = to_dev({k: (v[None] if torch.is_tensor(v) else v) for k, v in PEDataset([sid], cfg)[0].items()})
    maps, out = explain(model, b); sl = b["slice_lab"][0].cpu().numpy(); vm = b["vessel"]
    for name, mp in maps.items():
        z = mp[0, 0].amax((1, 2)).cpu().numpy(); zhit = float(sl[max(z.argmax() - 1, 0):z.argmax() + 2].max() > 0) if b["slice_valid"] > 0 else np.nan
        rows.append(dict(sid=sid, method=name, deletion=deletion_score(b, mp), z_hit=zhit, vessel_mass=float((mp * vm).sum() / mp.sum().clamp(min=1e-8))))
E = pd.DataFrame(rows).groupby("method").agg(deletion=("deletion", "mean"), z_hit=("z_hit", "mean"), vessel_mass=("vessel_mass", "mean")).round(3)
E.to_csv(f"{cfg.out_dir}/expE_explainability.csv"); E

## Uncertainty (MC-dropout) — calibration and selective prediction

In [ ]:
Ru = predict(model, make_loader(te, cfg, False), mc=20); y, p, sd = Ru["y"][:, 0], Ru["mc_mean"], Ru["mc_std"]
def ece(y, p, n=10):
    bins = np.minimum((p * n).astype(int), n - 1); return sum(abs(y[bins == k].mean() - p[bins == k].mean()) * (bins == k).mean() for k in range(n) if (bins == k).any())
print(f"AUC(MC mean)={safe_auc(y, p):.4f}  ECE={ece(y, p):.4f}  corr(std, |error|)={np.corrcoef(sd, np.abs(p - y))[0, 1]:.3f}")
order = np.argsort(sd); keep = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
print("selective prediction (retain most-certain fraction -> AUC):", {k: round(safe_auc(y[order[:int(k * len(y))]], p[order[:int(k * len(y))]]), 3) for k in keep})

## Experiment F — robustness
Perturbations are applied to the **test** volumes at inference time: image noise, simulated thicker slices (z-averaging), preprocessing/contrast shifts, in-plane blur; difficult subsets (QC-flagged, high-noise, low-contrast, low-burden); and class-imbalance stress (re-weighting prevalence).

In [ ]:
def noise(s):    return lambda v: (v + s * torch.randn_like(v)).clamp(0, 1)
def thick(f):    return lambda v: F.interpolate(F.avg_pool3d(v[None], (f, 1, 1), (f, 1, 1), ceil_mode=True), size=v.shape[1:], mode="trilinear", align_corners=False)[0]
def contrast(a, c): return lambda v: ((v - 0.5) * a + 0.5 + c).clamp(0, 1)
def blur(k):     return lambda v: F.avg_pool3d(v[None], (1, k, k), 1, (0, k // 2, k // 2))[0]
PERT = {"clean": None, "noise σ=0.02": noise(.02), "noise σ=0.05": noise(.05), "noise σ=0.10": noise(.10), "thick ×2": thick(2), "thick ×4": thick(4),
        "contrast ×0.8": contrast(.8, 0), "contrast ×1.2": contrast(1.2, 0), "shift +0.08": contrast(1, .08), "shift -0.08": contrast(1, -.08), "blur 3px": blur(3)}
rows = []
for n, fn in PERT.items():
    Rp = predict(model, make_loader(te, cfg, False, perturb=fn)); d = det_metrics(Rp)
    rows.append(dict(perturbation=n, auc=d["auc"], ap=d["ap"], slice_f1=slice_f1(Rp["slice_prob"], Rp["slice_lab"], Rp["slice_valid"], cfg.thr_slice)))
Fr = pd.DataFrame(rows).round(4); Fr.to_csv(f"{cfg.out_dir}/expF_robustness.csv", index=False); Fr

In [ ]:
qc = pd.read_csv(f"{cfg.out_dir}/qc.csv").set_index("study"); Rb = predict(model, make_loader(te, cfg, False)); sid = np.array(Rb["sid"])
q = qc.reindex(sid); y, p = Rb["y"][:, 0], Rb["det_prob"]; pe_pos = y > 0.5
subsets = {"all": np.ones(len(y), bool), "QC-flagged": q.flag.fillna(False).values, "high noise (top quartile)": (q.noise > q.noise.quantile(.75)).values,
           "low contrast (bottom quartile)": (q.contrast < q.contrast.quantile(.25)).values, "<10% burden PE+ (+all negatives)": ~pe_pos | (Rb["burden_t"] < 0.10)}
rows = [dict(subset=k, n=int(m.sum()), auc=safe_auc(y[m], p[m]), sens_at_thr=float((p[m & pe_pos] >= cfg.thr_det).mean()) if (m & pe_pos).any() else np.nan,
             spec_at_thr=float((p[m & ~pe_pos] < cfg.thr_det).mean()) if (m & ~pe_pos).any() else np.nan) for k, m in subsets.items()]
print(pd.DataFrame(rows).round(4).to_string(index=False))

rs = np.random.RandomState(0); P, N = np.where(pe_pos)[0], np.where(~pe_pos)[0]; rows = []
for prev in (0.01, 0.05, 0.10, 0.30, 0.50):                                             # class-imbalance stress test
    a, ap_, ppv = [], [], []
    for _ in range(200):
        nP = max(int(prev / (1 - prev) * len(N)), 2); idx = np.r_[rs.choice(P, min(nP, len(P)), replace=False), N]
        a.append(safe_auc(y[idx], p[idx])); ap_.append(average_precision_score(y[idx], p[idx])); h = p[idx] >= cfg.thr_det; ppv.append((y[idx][h] > .5).mean() if h.any() else np.nan)
    rows.append(dict(prevalence=prev, auc=np.nanmean(a), ap=np.nanmean(ap_), ppv=np.nanmean(ppv)))
print(pd.DataFrame(rows).round(4).to_string(index=False))

## Final explainable output for one study
Combines: CTPA slice · vessel map · localisation heat-map · Grad-CAM · vessel attention → **final explainable PE map**, with the embolus marked by a 3D bounding box and a structured report.

*Location wording is limited to what the pipeline can support*: side (left/right) and central vs. peripheral. Lobar/segmental naming (e.g. “right lower-lobe segmental artery”) requires a lobe/artery-branch labelling step that is **not** part of RSNA-STR; plug one in at `describe_location`.

In [ ]:
PATIENT_RIGHT_ON_IMAGE_LEFT = True      # standard radiological display; verify for your data orientation

def describe_location(cz, cy, cx):
    side = "right" if (cx < 0) == PATIENT_RIGHT_ON_IMAGE_LEFT else "left"
    return f"{side}-sided, {'central (near mediastinum)' if abs(cx) < 0.25 else 'peripheral'}"

def explain_case(sid, tau=0.5, mc=20):
    b = to_dev({k: (v[None] if torch.is_tensor(v) else v) for k, v in PEDataset([sid], cfg)[0].items()})
    maps, out = explain(model, b); p0 = float(torch.sigmoid(out["det_logit"].float()))
    enable_mc_dropout(model); ps = torch.stack([torch.sigmoid(model(b)["det_logit"].float()) for _ in range(mc)]); model.eval()
    comb = maps["combined"][0, 0].cpu().numpy(); vol = b["vol"][0, 0].cpu().numpy(); ves = b["vessel"][0, 0].cpu().numpy()
    lab, n = ndi.label(comb > tau); bbox = None; loc = "no embolus above threshold"
    if n:
        sums = ndi.sum(comb, lab, range(1, n + 1)); k = int(np.argmax(sums)) + 1; sl = ndi.find_objects(lab)[k - 1]; bbox = sl
        c = [(s.start + s.stop - 1) / 2 for s in sl]; D_, H_, W_ = comb.shape
        loc = describe_location(c[0] / (D_ - 1) * 2 - 1, c[1] / (H_ - 1) * 2 - 1, c[2] / (W_ - 1) * 2 - 1)
    bur = float(out["burden_ext"]); cat = ["low", "moderate", "high"][int(np.digitize(bur, cfg.burden_bins))]
    rv = float(torch.sigmoid(out["sev_logits"][0, 0]))
    txt = (f"PE {'DETECTED' if p0 >= cfg.thr_det else 'not detected'}  p={ps.mean():.2f} ± {ps.std():.2f}  |  location: {loc}\n"
           f"burden index (slice-extent) {bur:.2f} → {cat}  |  P(RV/LV≥1)={rv:.2f}   [research output — not for clinical use]")
    zc = int(round((bbox[0].start + bbox[0].stop - 1) / 2)) if bbox else int(np.argmax(comb.max((1, 2))))
    fig, ax = plt.subplots(1, 6, figsize=(21, 3.8)); ov = lambda a, mp, cm: (a.imshow(vol[zc], cmap="gray", vmin=0, vmax=1), a.imshow(np.ma.masked_where(mp[zc] < .15, mp[zc]), cmap=cm, alpha=.6, vmin=0, vmax=1))
    ax[0].imshow(vol[zc], cmap="gray", vmin=0, vmax=1); ax[0].set_title("CTPA slice")
    ov(ax[1], ves, "autumn"); ax[1].set_title("vessel map")
    ov(ax[2], maps["heat"][0, 0].cpu().numpy(), "jet"); ax[2].set_title("PE localisation")
    ov(ax[3], maps["gradcam"][0, 0].cpu().numpy(), "jet"); ax[3].set_title("Grad-CAM")
    ov(ax[4], maps["attention"][0, 0].cpu().numpy(), "jet"); ax[4].set_title("vessel attention")
    ov(ax[5], comb, "jet"); ax[5].set_title("final explainable PE map")
    if bbox is not None:
        for a in (ax[0], ax[5]): a.add_patch(plt.Rectangle((bbox[2].start, bbox[1].start), bbox[2].stop - bbox[2].start, bbox[1].stop - bbox[1].start, fill=False, ec="cyan", lw=2))
    for a in ax: a.axis("off")
    plt.suptitle(f"{sid}\n{txt}", fontsize=10, y=1.12); plt.tight_layout(); plt.show(); print(txt)

for sid in pos_ids[:3]: explain_case(sid)